In [4]:
"""
FEES STRUCTURE

| Component       | Cost                                      | Your ETFs                |
| --------------- | ----------------------------------------- | ------------------------ |
| IBKR Commission | 0.05% of trade value (min €1.50, max €49) | All liquid ETFs          |
| Exchange Fee    | CHF1.50 + 0.015% (EBS/SIX)                | ~€1.60 + 0.015%          |
| Clearing Fee    | CHF0.38 flat                              | ~€0.40                   |
| Total Fixed     | ~€3.50 minimum                            | Per trade                |
| Total %         | 0.065% + €3.50                            | For €10k trade = 11.5bps |

Formula: cost = max(€3.50, 0.00065 * trade_value)

| ETF           | Typical Spread | Half-Spread Cost |
| ------------- | -------------- | ---------------- |
| CSPX (S&P500) | 0.01-0.02%     | 0.75bps          |
| EIMI (EM)     | 0.08-0.12%     | 5bps             |
| CNYA (China)  | 0.15-0.25%     | 10bps            |
| INR (India)   | 0.20-0.35%     | 13bps            |
| Gold (XAD5)   | 0.05-0.08%     | 3.5bps           |
| Average       | 0.10%          | 6.5bps           |

"""

'\nFEES STRUCTURE\n\n| Component       | Cost                                      | Your ETFs                |\n| --------------- | ----------------------------------------- | ------------------------ |\n| IBKR Commission | 0.05% of trade value (min €1.50, max €49) | All liquid ETFs          |\n| Exchange Fee    | CHF1.50 + 0.015% (EBS/SIX)                | ~€1.60 + 0.015%          |\n| Clearing Fee    | CHF0.38 flat                              | ~€0.40                   |\n| Total Fixed     | ~€3.50 minimum                            | Per trade                |\n| Total %         | 0.065% + €3.50                            | For €10k trade = 11.5bps |\n\nFormula: cost = max(€3.50, 0.00065 * trade_value)\n\n| ETF           | Typical Spread | Half-Spread Cost |\n| ------------- | -------------- | ---------------- |\n| CSPX (S&P500) | 0.01-0.02%     | 0.75bps          |\n| EIMI (EM)     | 0.08-0.12%     | 5bps             |\n| CNYA (China)  | 0.15-0.25%     | 10bps            |\n| INR

In [5]:
import numpy as np

def transaction_costs(trade_values, etf_spreads, portfolio_value):
    """
    trade_values: array of € amounts traded per ETF
    etf_spreads: half-spread % per ETF
    """
    # 1. Brokerage (IBKR tiered: 0.05% min €3.50)
    brokerage = np.maximum(3.50, 0.0005 * trade_values)
    
    # 2. Exchange + clearing (~€1.90 per trade)
    exchange = 1.90 * (trade_values > 0)
    
    # 3. Spread costs (round-trip = 2 * half-spread)
    spreads = 2 * etf_spreads * trade_values
    
    total_costs = brokerage + exchange + spreads
    return np.sum(total_costs) / portfolio_value  # % drag

# ETFs spreads (bps) - estimated
etf_spreads = np.array([0.0075, 0.050, 0.100, 0.035, 0.130, 0.030, 0.030]) / 100

# # In backtest:
# prev_weights = np.ones(8) / 8  # initial equal weight
# total_tc_costs = []

# for w_opt in weights_opt:
#     tc_cost, turnover = transaction_costs(prev_weights, w_opt, 100000, etf_spreads)
#     total_tc_costs.append(tc_cost)
#     returns_with_costs = returns - tc_cost  # subtract from period return
#     prev_weights = w_opt

def realistic_rebalance(portfolio_value, target_weights, prev_weights, etf_prices, prev_shares, weight_diff_rebalance):
    """
    prev_shares: (N,) current shares held (persistent across calls)
    Returns: new_shares, trade_values, tc_drag, actual_weights
    """

    diff = np.abs(prev_weights - target_weights)
    
    # Target values excluding cash
    target_etf_values = portfolio_value * target_weights[:-1]
    
    # Target shares (not rounded)
    target_shares = target_etf_values / etf_prices
    
    # Trades required
    shares_traded = np.abs(target_shares - prev_shares)
    trade_values = shares_traded * etf_prices

    if diff[diff >= weight_diff_rebalance].any():
        return target_shares, trade_values, 0.0, target_weights, portfolio_value
    
    # Transaction cost drag
    tc_drag = transaction_costs(trade_values, etf_spreads, portfolio_value)
    
    # Actual post-trade portfolio value (after costs)
    new_portfolio_value = portfolio_value - tc_drag * portfolio_value
    
    # Actual weights (including rounding error)
    actual_etf_values = target_shares * etf_prices
    cash_value = new_portfolio_value - np.sum(actual_etf_values)
    actual_weights = np.append(actual_etf_values / new_portfolio_value, cash_value / new_portfolio_value)
    actual_weights = np.array(actual_weights)
    
    return target_shares, trade_values, tc_drag, actual_weights, new_portfolio_value

In [6]:
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go

snp = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CSPX ETF Stock Price History.csv")
china = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CNYA ETF Stock Price History.csv")
em = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "EIMI ETF Stock Price History.csv")
gold = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XAD5 ETF Stock Price History.csv")
india = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "INR ETF Stock Price History.csv")
mscieurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XMEU ETF Stock Price History.csv")
smallcapeurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "SXRJ ETF Stock Price History.csv")

dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope]
date_sets = [set(df["Date"]) for df in dfs]

# Find the intersection of all date sets
common_dates = set.intersection(*date_sets)

# Filter each DataFrame to include only common dates
dfs = [df[df["Date"].isin(common_dates)] for df in dfs]

N = len(dfs)
T = len(dfs[0]) - 1

def correct_volume(vol_value):
    if isinstance(vol_value, (float, int)):
        return vol_value
    if vol_value[-1] == "K":
        return 1000 * float(vol_value[:-1])
    if vol_value[-1] == "M":
        return 1_000_000 * float(vol_value[:-1])
    return float(vol_value)

returns_all = np.empty((T, N))
sorted_returns_all = np.empty((T, N))
volumes_all = np.empty((T, N))
prices_all = np.empty((T, N))
dfs_new = []
for i, df in enumerate(dfs):
    df = df[::-1]
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce")  # Handles str/NaN/object
    returns = df["Price"].pct_change().values[1:]
    returns_all[:, i] = returns
    sorted_returns_all[:, i] = sorted(returns)
    volumes_all[:, i] = df["Vol."].apply(correct_volume).values[1:]
    prices_all[:,i] = df["Price"].values[1:]
    dfs_new.append(df)

all_dates = dfs[0]["Date"].values[1:]
volumes_all = np.nan_to_num(volumes_all, nan=0.0)

In [7]:
graph = go.Figure()
for i in range(returns_all.shape[1]):
    L = list(range(T))
    cum_returns = np.cumprod(1 + returns_all[:,i]) - 1
    graph.add_trace(go.Scatter(x=df["Date"], y=cum_returns))
graph.show()

In [8]:
import tensorflow as tf

def max_drawdown(prices, weights):
    # prices (T,N)
    # weights (N,)
    T, N = prices.shape
    portfolio_values = prices @ weights # (T,)
    highest_peak = -1
    worst_pct = 0.0
    for t in range(T):
        highest_peak = max(highest_peak, portfolio_values[t])
        worst_pct = min(worst_pct, portfolio_values[t] / highest_peak - 1)
    return worst_pct

def perform_validation(w, validation_data, validation_prices):
    # validation data is shape (T, N)

    cash_returns = np.zeros((validation_data.shape[0], 1))  # or use risk-free rate
    validation_data_with_cash = np.hstack([validation_data, cash_returns])
    #print('shape of validation_data_with_cash:', validation_data_with_cash.shape)
    #print('weights shape:', w.shape)

    # Reshape w to (8, 1) for matrix multiplication
    w = tf.reshape(w, (-1, 1))

    # perform metrics
    total_return = validation_data_with_cash @ w  # shape (T, 1)
    total_return = total_return.numpy().flatten()  # convert to (T,)

    cumulative_returns = np.cumprod(total_return + 1) - 1
    risk_free = 0.0
    std = np.std(total_return)
    downside_std = np.std(total_return[total_return < 0])
    sharpe = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * std)
    sortino = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * downside_std)

    alpha = 0.10

    var_alpha = np.quantile(total_return, alpha)
    cvar = total_return[total_return <= var_alpha].mean()
    mean_return = np.mean(total_return)

    md = max_drawdown(validation_prices, w[:-1].numpy())

    return var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
    

C:\Users\tobia\AppData\Roaming\Python\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning:

Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.

C:\Users\tobia\AppData\Roaming\Python\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning:

Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.

C:\Users\tobia\AppData\Roaming\Python\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning:

Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update the gencode to avoid c

In [26]:
print(max_drawdown(prices_all, np.ones(N)/N))

-0.28127336452283547


In [14]:
def lower_part_mean(matrix):
    n, _ = matrix.shape
    s = 0.0
    for i in range(1,n):
        for j in range(i):
            s += matrix[i,j]
    N = (n-1)*n/2
    return s / N

def compute_statistics_rolling(returns, window_size, stepsize):
    print('returns:'); print(returns)
    all_corrs = []
    all_volas = []
    T, N = returns.shape
    w = np.ones(N) / N
    #print('T:', T)
    #print('testetst')
    for idx in range(0, T - window_size, stepsize):
        section = returns[idx:idx+window_size,:]
        #print('section shape:', section.shape)
        corr = np.corrcoef(section, rowvar=False) # (N,N)
        #print('appending', lower_part_mean(corr))
        all_corrs.append(lower_part_mean(corr))
        returns_total = section @ w
        #print('returns_total:', returns_total)
        std = np.std(returns_total)
        all_volas.append(std)
    
    all_corrs = np.array(all_corrs)
    all_volas = np.array(all_volas)
    #print('result:'); print(all_corrs)
    return all_corrs, all_volas


In [16]:
# run

corrs, volas = compute_statistics_rolling(returns_all, 90, 30)

print('mean corrs:', np.mean(corrs))
print('mean volas:', np.mean(volas))
print('highest corrs:', np.max(corrs))
print('lowest corrs:', np.min(corrs))
print('lowest volas:', np.min(volas))
print('highest volas:', np.max(volas))
# results:
# based on this, determine threshold for correlation and volatility

vola_thresh = np.percentile(volas, 75)
corr_thresh = np.percentile(corrs, 75)

corrs_mean = np.mean(corrs)
corrs_std = np.std(corrs)
volas_mean = np.mean(volas)
volas_std = np.std(volas)

returns:
[[ 0.00583242 -0.00188324 -0.00078462 ...  0.00178147  0.00061637
   0.00415324]
 [ 0.00041419  0.03962264  0.01177856 ... -0.02489627 -0.00517433
  -0.01030928]
 [-0.0100916  -0.03085299 -0.01590997 ... -0.01823708 -0.0152322
  -0.01896208]
 ...
 [-0.00759045 -0.01013514 -0.00854345 ... -0.01162325 -0.00697306
   0.00134771]
 [-0.0127289  -0.01194539 -0.01723413 ... -0.01662612 -0.02734333
  -0.02706744]
 [-0.00528727 -0.01208981 -0.01796407 ... -0.01773196 -0.01154014
  -0.01629265]]
mean corrs: 0.3823000782738932
mean volas: 0.008017332792505716
highest corrs: 0.6354282873413744
lowest corrs: 0.18268267958087434
lowest volas: 0.003908344756921926
highest volas: 0.021881164993669322


In [ ]:
import tensorflow as tf

alpha = 0.05   # tail probability for CVaR
lambda_ = 0.5  # CVaR penalty
learning_rate = 0.01
gamma = 0.03
num_steps = 2000
beta = 0.1 # penalty volatility
min_weight = 1/9 # 1/N = 1/7, so a little less
max_weight = 0.3
tau = 0.5 # to start
a = 1.0 # for stress environment tuning
b = 1.0 # for stress environment tuning
weight_diff_rebalance = 0.10 # minimum weight difference for a portfolio rebalance
portfolio_value = 10_000

def sigmoid(x):
    return 1/(1+np.exp(-x))

def normalize(x):
    return (x - tf.reduce_mean(x)) / (tf.math.reduce_std(x) + 1e-8)

def rank_transform(x):
    ranks = tf.argsort(tf.argsort(x))
    return tf.cast(ranks, tf.float32) / tf.cast(tf.size(x), tf.float32)

def gradient(train_data, vol_data):

    # returns (weights, mode, had_to_cap_weights, stress_factor, turnover_values)
    # where mode = 0 if returned normal (optimal) weights,
    # 1 = if returned equal because of correlations
    # 2 = if returned equal because of volatility

    # train data: (T,N)
    w = tf.Variable(tf.ones([N], dtype=tf.float32) / N)
    t = tf.Variable(0.0, dtype=tf.float32)  # VaR threshold

    optimizer = tf.keras.optimizers.Adam(learning_rate)

    turnover_values = []

    X_tf = tf.convert_to_tensor(train_data, dtype=tf.float32) # (T,N)
    V_tf = tf.convert_to_tensor(vol_data, dtype=tf.float32) # (T,N)
    #cov_tf = tf.convert_to_tensor(cov, dtype=tf.float32)  # shape (N, N)

    corr = np.corrcoef(train_data, rowvar=False)
    mean_corr = lower_part_mean(corr)
    #print('mean corr:', mean_corr)
    corr_standardized = (mean_corr - corrs_mean) / corrs_std

    w_prev = tf.identity(w)

    for step in range(num_steps):
        with tf.GradientTape() as tape:
            # enforce sum-to-1 via softmax
            w_normalized = tf.nn.softmax(w)
            w_col = tf.reshape(w_normalized, [N, 1])  # shape (N,1)

            # portfolio returns
            portfolio_returns = tf.matmul(X_tf, tf.reshape(w_col, [-1,1])) # (T)
            losses = -portfolio_returns
            cvar = t + (1 / (1 - alpha)) * tf.reduce_mean(tf.nn.relu(losses - t))

            # momentum per asset (N,)
            momentum = tf.reduce_mean(X_tf[-60:], axis=0)

            # volume per asset (N,)
            vol_short = tf.reduce_mean(V_tf[-10:], axis=0)
            vol_long  = tf.reduce_mean(V_tf[-60:], axis=0)
            volume_signal = (vol_short - vol_long) / (vol_long + 1e-8)

            volatility = tf.math.reduce_std(X_tf[-60:], axis=0)
            risk_adj_momentum = momentum / (volatility + 1e-8)

            m1 = normalize(momentum)
            m2 = normalize(risk_adj_momentum)
            m3 = normalize(volume_signal)
            #print('m1, m2, m3:', m1, m2, m3)

            # combine (N,)
            mu = 0.5 * m1 + 0.3 * m2 + 0.2 * m3
            mu = rank_transform(mu)

            turnover_penalty = tau * tf.reduce_sum(tf.square(w - w_prev))
            turnover = tf.reduce_sum(
                tf.abs(w_normalized - w_prev)
            ).numpy()
            turnover_values.append(turnover)

            expected_return = tf.tensordot(w_normalized, mu, axes=1)
            weights_penalty = gamma * tf.reduce_sum(tf.math.square(w_col))

            market_return = tf.reduce_mean(X_tf[-20:])   # avg across assets
            market_vol = tf.math.reduce_std(X_tf[-20:])

            objective = -(expected_return - lambda_ * cvar) + weights_penalty + turnover_penalty

        # compute gradients
        grads = tape.gradient(objective, [w, t])
        optimizer.apply_gradients(zip(grads, [w, t]))

        w_prev = tf.identity(w)

    # after the optimization loop
    optimal_w = tf.nn.softmax(w).numpy()

    # compute volatility
    combined_returns = train_data @ optimal_w # (T,)
    volatility = np.std(combined_returns)
    volatility_standardized = (volatility - volas_mean) / volas_std

    stress_score = sigmoid(a * corr_standardized + b * volatility_standardized)
    chosen_w = stress_score * np.ones(N) / N + (1 - stress_score) * optimal_w

    crash_signal = tf.sigmoid(-5 * market_return + 5 * market_vol)
    market_return = tf.reduce_mean(X_tf[-20:])   # avg across assets
    crash_signal = tf.sigmoid(-5 * market_return + 5 * market_vol)
    cash_weight = crash_signal * 0.5  # up to 50% cash
    w_risky = (1 - cash_weight) * chosen_w
    w_final = tf.concat([w_risky, [cash_weight]], axis=0)

    if tf.reduce_any(w_final <= min_weight):
        w_final = np.clip(w_final, a_min = min_weight, a_max = max_weight)
        return w_final / np.sum(w_final), True, stress_score, np.mean(turnover_values)
    return w_final.numpy(), False, stress_score, np.mean(turnover_values)

def rolling_window(window_size, validation_size, stepsize, returns_data, volume_data, price_data, portfolio_init=10_000):
    # data is (T,N)

    sharpes_w_opt= []
    sharpes_w_eqw = []

    mean_return_w_opt = []
    mean_return_w_eqw = []

    sortinos_w_opt = []
    sortinos_w_eqw = []

    vars_w_opt = []
    vars_w_eqw = []

    cvars_w_opt = []
    cvars_w_eqw = []

    mds_w_opt = []
    mds_w_eqw = []

    stress_numbers = []
    turnover_all = []

    T, N = returns_data.shape
    weights_opt = []

    clip_counter = 0
    total_counter = 0

    portfolio_values = [portfolio_init]
    prev_shares = np.zeros(N)  # Start with no positions
    tc_drags = [] # drag trading costs (pct)
    w_prev = np.ones(N+1)/(N+1)  # Start with equal weights

    for idx in range(0, len(returns) - window_size - validation_size, stepsize):
        train_data_returns = returns_data[idx:idx+window_size]
        train_data_volume = volume_data[idx:idx+window_size]
        train_data_prices = price_data[idx:idx+window_size]

        val_data_returns = returns_data[idx+window_size:idx+window_size+validation_size]
        val_data_prices = price_data[idx+window_size:idx+window_size+validation_size]

        w, applied_clip, stress, turnovers = gradient(train_data_returns, train_data_volume)
        weights_opt.append(w)
        if applied_clip:
            clip_counter += 1
        total_counter += 1
        turnover_all.append(turnovers)
        stress_numbers.append(stress)

        # Prices at rebalance (last train day)
        rebalance_prices = train_data_prices[-1]
        current_value = portfolio_values[-1]
        
        # Execute rebalance
        new_shares, trade_vals, tc_drag, actual_weights, new_value = realistic_rebalance(
            current_value, w, w_prev, rebalance_prices, prev_shares, weight_diff_rebalance
        ) # trade vals = dollar values of etfs, new_value = new portfolio balance

        prev_shares = new_shares[:]
        portfolio_values.append(new_value)
        tc_drags.append(tc_drag)

        ( # replaced w with actual_weights
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
        ) = perform_validation(actual_weights, val_data_returns, val_data_prices)

        sharpes_w_opt.append(sharpe)
        sortinos_w_opt.append(sortino)
        vars_w_opt.append(var_alpha)
        cvars_w_opt.append(cvar)
        mean_return_w_opt.append(mean_return)
        mds_w_opt.append(md)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
        ) = perform_validation(np.ones(N+1) / (N+1), val_data_returns, val_data_prices)

        sharpes_w_eqw.append(sharpe)
        sortinos_w_eqw.append(sortino)
        vars_w_eqw.append(var_alpha)
        cvars_w_eqw.append(cvar)
        mean_return_w_eqw.append(mean_return)
        mds_w_eqw.append(md)

        w_prev = actual_weights.copy()

    df = pd.DataFrame({
        "Sharpes": [np.mean(sharpes_w_eqw), np.mean(sharpes_w_opt)],
        "Sortinos": [np.mean(sortinos_w_eqw), np.mean(sortinos_w_opt)],
        "Vars": [np.mean(vars_w_eqw), np.mean(vars_w_opt)],
        "Cvars": [np.mean(cvars_w_eqw), np.mean(cvars_w_opt)],
        "MeanReturn": [np.mean(mean_return_w_eqw), np.mean(mean_return_w_opt)],
        "Highest maximum drawdowns": [np.min(mds_w_eqw), np.min(mds_w_opt)],
        "Mean maximum drawdowns": [np.mean(mds_w_eqw), np.mean(mds_w_opt)]
    }, index=["Equal weights", "Return + CVaR"])

    weights_opt = np.array(weights_opt)

    return df, np.mean(weights_opt, axis=0), weights_opt, clip_counter / total_counter, stress_numbers, turnover_all, tc_drags

In [ ]:
df, weights_opt_mean, weights_opt, clip_pct, stresses, turnover, costs = rolling_window(
    window_size=252*2,
    validation_size=252,
    stepsize=252,
    returns_data=returns_all,
    volume_data=volumes_all,
    price_data=prices_all
)

print(df)

,Sharpes,Sortinos,Vars,Cvars,MeanReturn,Highest maximum drawdowns,Mean maximum drawdowns
Equal weights,0.881774,1.125267,-0.007584,-0.013134,0.000386,-0.238112,-0.115774
Return + CVaR,0.957290,1.256047,-0.006015,-0.010336,0.000311,-0.223493,-0.112572


In [36]:
print('chosen weights')
print(weights_opt)
print()
print('mean weights:')
print(weights_opt_mean)
print()
print('clipped pct:', clip_pct)
print()
print('stresses:'); print(stresses)
print('turnovers:'); print(turnover)
print()
print('costs:')
print(costs)
print('total cost (pct):', np.sum(costs))


chosen weights
[[0.0870698  0.0870698  0.0870698  0.0870698  0.0870698  0.0870698
  0.23178898 0.24579223]
 [0.06955424 0.06955424 0.06955424 0.3859012  0.06955424 0.06955424
  0.06955424 0.19677332]
 [0.06998118 0.06998118 0.06998118 0.37885737 0.06998118 0.06998118
  0.06998118 0.2012556 ]
 [0.10084654 0.10084315 0.10085278 0.10084946 0.10092708 0.10084834
  0.13734314 0.2574895 ]
 [0.09225186 0.09224334 0.09223597 0.1846313  0.09244138 0.09224053
  0.09222746 0.26172814]
 [0.0838998  0.0838998  0.0838998  0.0838998  0.0838998  0.0838998
  0.26035553 0.2362456 ]
 [0.31414872 0.07803108 0.07803108 0.07803108 0.07803108 0.07803108
  0.07803108 0.2176648 ]
 [0.06500465 0.06500465 0.06500465 0.06500465 0.06500465 0.42604652
  0.06500465 0.1839256 ]]

mean weights:
[0.1103446  0.08082841 0.08082869 0.17053059 0.08086365 0.12595893
 0.12553579 0.22510937]

clipped pct: 0.75

stresses:
[np.float64(0.7859107780187802), np.float64(0.37172248415598513), np.float64(0.38654260084707626), np.floa

In [32]:
weights_graph = go.Figure()
L = list(range(len(weights_opt)))
for i in range(len(weights_opt[0])):
    to_plot = [w[i] for w in weights_opt]
    weights_graph.add_trace(go.Scatter(x=L, y=to_plot))
weights_graph.show()

In [34]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=1,cols=2)
fig.add_trace(go.Scatter(x=list(range(len(stresses))), y=stresses, name="Stress"), row=1, col=1)
fig.add_trace(go.Scatter(x=list(range(len(stresses))), y=costs, name="Costs"), row=1, col=2)
fig.show()


In [2]:
def stress_test(df_data, start_train, end_train, end_valid):

    val_returns_total = []
    train_returns_total = []

    train_volume_total = []
    val_volume_total = []

    val_prices_total = []

    for df in df_data:
        df_datetime = df.copy()
        df_datetime["Date"] = pd.to_datetime(df_datetime["Date"])
        train = df_datetime[
            (df_datetime["Date"] >= pd.to_datetime(start_train))
            & (df_datetime["Date"] <= pd.to_datetime(end_train))]
        val = df_datetime[
            (df_datetime["Date"] >= pd.to_datetime(end_train))
            & (df_datetime["Date"] <= pd.to_datetime(end_valid))
        ]

        train_returns = train["Price"].pct_change().values[1:]
        val_returns = val["Price"].pct_change().values[1:]

        val_returns_total.append(val_returns)
        train_returns_total.append(train_returns)

        train_volume_total.append(np.nan_to_num(train["Vol_clean"].values[1:], nan=0.0))
        val_volume_total.append(np.nan_to_num(val["Vol_clean"].values[1:], nan=0.0))

        val_prices_total.append(val["Price"].values[1:])
        
    train_returns_total = np.array(train_returns_total).T # shape (num_days, N = num_assets)
    val_returns_total = np.array(val_returns_total).T
    train_volume_total = np.array(train_volume_total).T
    val_volume_total = np.array(val_volume_total).T
    val_prices_total = np.array(val_prices_total).T

    w_opt, cut_weights, stress_score, mean_turnover = gradient(train_returns_total, train_volume_total)
    
    return w_opt, perform_validation(w_opt, val_returns_total, val_prices_total), perform_validation(np.ones(N + 1) / (N+1), val_returns_total, val_prices_total), cut_weights, stress_score, mean_turnover


In [38]:
start_train = "2019-10-01"
end_train = "2020-02-01"
end_valid = "2020-03-01"

chosen_weights, results_optimized, results_eqw, cut_weights, stress, turnover_values = stress_test(
    dfs_new, start_train, end_train, end_valid
)

var_alpha, cvar, sharpe, sortino, cumulative_returns_opt, mean_return, md = results_optimized

print('OPTIMIZED WEIGHTS RESULTS')
print('chosen weights:', chosen_weights)
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print('capped weights:', cut_weights)
print()

var_alpha, cvar, sharpe, sortino, cumulative_returns_eqw, mean_return, md = results_eqw

print('EQUAL WEIGHTS RESULTS')
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print()

print('turnover values:', turnover_values)
print('stress:', stress)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_eqw))), y=cumulative_returns_eqw, name="equal weights"))
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_opt))), y=cumulative_returns_opt, name="optimized weights"))
fig.show()


OPTIMIZED WEIGHTS RESULTS
chosen weights: [0.32297352 0.07654864 0.07654864 0.07654864 0.07654864 0.07654864
 0.07654864 0.21773472]
var: -0.021802772
cvar: -0.025994716
sharpe: -4.178692785796754
sortino: -4.675579183217794
mean return: -0.0031764486
maximum drawdown: [-0.10982902]
capped weights: True

EQUAL WEIGHTS RESULTS
var: -0.02201177228372069
cvar: -0.027318205932842975
sharpe: -2.9461734725881743
sortino: -3.6141846800277087
mean return: -0.0024002768880687403
maximum drawdown: [-0.10527719]

turnover values: 15.575605
stress: 0.5611863846439302


In [17]:
start_train = "2021-02-22"
end_train = "2022-01-01"
end_valid = "2023-01-23"

chosen_weights, results_optimized, results_eqw, cut_weights, stress, turnover_values = stress_test(
    dfs_new, start_train, end_train, end_valid
)

var_alpha, cvar, sharpe, sortino, cumulative_returns_opt, mean_return, md = results_optimized

print('OPTIMIZED WEIGHTS RESULTS')
print('chosen weights:', chosen_weights)
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print('capped weights:', cut_weights)
print()

var_alpha, cvar, sharpe, sortino, cumulative_returns_eqw, mean_return, md = results_eqw

print('EQUAL WEIGHTS RESULTS')
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print()

print('turnover values:', turnover_values)
print('stress:', stress)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_eqw))), y=cumulative_returns_eqw, name="equal weights"))
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_opt))), y=cumulative_returns_opt, name="optimized weights"))
fig.show()


OPTIMIZED WEIGHTS RESULTS
chosen weights: [0.28744912 0.08085024 0.08085024 0.08085024 0.08085024 0.08085024
 0.08085024 0.22744945]
var: -0.00964557
cvar: -0.014376231
sharpe: -0.4500085594528203
sortino: -0.7032804925903947
mean return: -0.00022102868
maximum drawdown: [-0.16956557]
capped weights: True

EQUAL WEIGHTS RESULTS
var: -0.00996141798785949
cvar: -0.01455306277674845
sharpe: -0.3541397363320231
sortino: -0.5500624561361799
mean return: -0.00017931253940894798
maximum drawdown: [-0.15853415]

turnover values: 15.560294
stress: 0.6585143272094391


In [18]:
start_train = "2024-01-01"
end_train = "2025-01-01"
end_valid = "2025-04-23"

chosen_weights, results_optimized, results_eqw, cut_weights, stress, turnover_values = stress_test(
    dfs_new, start_train, end_train, end_valid
)

var_alpha, cvar, sharpe, sortino, cumulative_returns_opt, mean_return, md = results_optimized

print('OPTIMIZED WEIGHTS RESULTS')
print('chosen weights:', chosen_weights)
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print('capped weights:', cut_weights)
print()

var_alpha, cvar, sharpe, sortino, cumulative_returns_eqw, mean_return, md = results_eqw

print('EQUAL WEIGHTS RESULTS')
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print()

print('turnover values:', turnover_values)
print('stress:', stress)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_eqw))), y=cumulative_returns_eqw, name="equal weights"))
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_opt))), y=cumulative_returns_opt, name="optimized weights"))
fig.show()


OPTIMIZED WEIGHTS RESULTS
chosen weights: [0.42032057 0.06568063 0.06568063 0.06568063 0.06568063 0.06568063
 0.06568063 0.18559557]
var: -0.010649668
cvar: -0.020394247
sharpe: -0.43081959163420214
sortino: -0.5234587779555989
mean return: -0.0002730429
maximum drawdown: [-0.13885679]
capped weights: True

EQUAL WEIGHTS RESULTS
var: -0.008589521683481635
cvar: -0.01787215907567255
sharpe: 0.2426532499315051
sortino: 0.27948577374191824
mean return: 0.00013818415223937935
maximum drawdown: [-0.12808161]

turnover values: 15.557263
stress: 0.25001041780141625
